# Nested Pandas with Hyrax

This notebook creates a small nested-pandas parquet file with flat scalar columns and a nested light-curve column, then reads it with `NestedPandasDataset`.

In [ ]:
from pathlib import Path

import pandas as pd
import nested_pandas as npd
import hyrax

In [ ]:
# Build a NestedFrame with three objects, each with a two-point light curve.
flat = npd.NestedFrame(
    {
        "object_id": ["obj_0", "obj_1", "obj_2"],
        "ra": [10.0, 20.0, 30.0],
        "dec": [5.0, 6.0, 7.0],
    }
)

light_curves = pd.DataFrame(
    {
        "time": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
        "flux": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
        "object_id": ["obj_0", "obj_0", "obj_1", "obj_1", "obj_2", "obj_2"],
    }
).set_index("object_id")

nf = flat.set_index("object_id").join_nested(light_curves, "lc")
nf

In [ ]:
# Write to parquet.
data_path = Path("./sample_nested.parquet")
nf.to_parquet(str(data_path))

In [ ]:
# Load with Hyrax.
h = hyrax.Hyrax()
h.config["data_request"] = {
    "infer": {
        "sources": {
            "dataset_class": "NestedPandasDataset",
            "data_location": str(data_path),
            "primary_id_field": "object_id",
            "fields": ["object_id", "ra", "dec", "lc"],
        }
    }
}
prepared = h.prepare()

In [ ]:
# Inspect a single sample — flat fields are scalars, nested field is a DataFrame.
prepared["infer"][0]["sources"]